<a href="https://colab.research.google.com/github/Aleksandr34nov/ProductAndCategoryPairs/blob/main/ProductAndCategoryPairs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import random

class ProductCategoryGenerator:
    """
    Класс для формирования датафреймов и получения тогового дата фрейма «Имя продукта – Имя категории»
    """
    # Конструктор класса
    def __init__(self, num_products=5, num_categories=3):
        self.num_products = num_products
        self.num_categories = num_categories
        self.spark = SparkSession.builder.appName("RandomProductCategoryGeneration").getOrCreate()
        self.products_df = None
        self.categories_df = None
        self.product_category_df = None
    # Метод для случайной генерации продуктов и категорий, а также связей между ними
    def generate_data(self):
        products_data = [(i, f"Product {i}") for i in range(1, self.num_products + 1)]
        categories_data = [(i, f"Category {i}") for i in range(1, self.num_categories + 1)]

        self.products_df = self.spark.createDataFrame(products_data, ["product_id", "product_name"])
        self.categories_df = self.spark.createDataFrame(categories_data, ["category_id", "category_name"])

        # Генерация случайных связей между продуктами и категориями
        product_category_data = []
        for product_id in range(1, self.num_products + 1):
            num_assigned_categories = random.randint(0, self.num_categories)
            assigned_categories = random.sample(range(1, self.num_categories + 1), num_assigned_categories)
            for category_id in assigned_categories:
                product_category_data.append((product_id, category_id))

        self.product_category_df = self.spark.createDataFrame(product_category_data, ["product_id", "category_id"])

    # Метод для получения всех пар «Имя продукта – Имя категории»
    def get_product_categories_dataframe(self):
        product_category_pairs = self.products_df.join(self.product_category_df, on="product_id", how="inner").join(self.categories_df, on="category_id", how="inner").select("product_name", "category_name")
        products_without_categories = self.products_df.join(self.product_category_df, on="product_id", how="left_anti").select("product_name", F.lit("Без категории").alias("category_name"))
        result_df = product_category_pairs.union(products_without_categories)
        return product_category_pairs, products_without_categories, result_df # Возвращаем датафреймы: продукты с категориями, без категорий и общий датафрейм

    # Метод для вывода результатов
    def show(self):
        product_category_pairs, products_without_categories, result_df = self.get_product_categories_dataframe()
        print("Общий датафрейм:")
        result_df.show()

        print("Датафрейм продуктов с категориями:")
        product_category_pairs.show()

        print("Датафрейм продуктов без категорий:")
        products_without_categories.show()

    # Метод для остановки SparkSession
    def stop_spark(self):
        if self.spark:
            self.spark.stop()

if __name__ == "__main__":
    generator = ProductCategoryGenerator(num_products=5, num_categories=3)
    generator.generate_data()
    generator.show()
    generator.stop_spark()


Общий датафрейм:
+------------+-------------+
|product_name|category_name|
+------------+-------------+
|   Product 5|   Category 1|
|   Product 4|   Category 1|
|   Product 4|   Category 3|
|   Product 5|   Category 2|
|   Product 4|   Category 2|
|   Product 1|Без категории|
|   Product 3|Без категории|
|   Product 2|Без категории|
+------------+-------------+

Датафрейм продуктов с категориями:
+------------+-------------+
|product_name|category_name|
+------------+-------------+
|   Product 5|   Category 1|
|   Product 4|   Category 1|
|   Product 4|   Category 3|
|   Product 5|   Category 2|
|   Product 4|   Category 2|
+------------+-------------+

Датафрейм продуктов без категорий:
+------------+-------------+
|product_name|category_name|
+------------+-------------+
|   Product 1|Без категории|
|   Product 3|Без категории|
|   Product 2|Без категории|
+------------+-------------+

